# Notebook 2 — GAN Experiments
Compare sample quality across Vanilla GAN → DCGAN → Conditional GAN → CycleGAN.

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from pathlib import Path
from PIL import Image
import torchvision.transforms as T

# Load and display training progress grids for each phase
phases = ['vanilla_gan', 'dcgan', 'cgan']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, phase in zip(axes, phases):
    # Load last saved epoch grid
    grids = sorted(Path(f'outputs/{phase}').glob('epoch_*.png'))
    if grids:
        img = Image.open(grids[-1])
        ax.imshow(img)
        ax.set_title(f'{phase.upper()}\n(Epoch {grids[-1].stem.split("_")[-1]})', fontsize=11)
    else:
        ax.text(0.5, 0.5, f'{phase}\n(not trained yet)', ha='center', va='center')
    ax.axis('off')

fig.suptitle('Generator Output Quality Across Phases', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/phase_comparison.png', dpi=150)
plt.show()

In [ ]:
# Generate fresh conditional samples — one row per DR grade
import sys; sys.path.insert(0, '..')
from models.cgan import Generator, LATENT_DIM, N_CLASSES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G = Generator().to(device)

ckpts = sorted(Path('checkpoints/cgan').glob('G_epoch_*.pt'))
if ckpts:
    G.load_state_dict(torch.load(ckpts[-1], map_location=device))
    G.eval()

    fig, axes = plt.subplots(5, 8, figsize=(16, 10))
    grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

    with torch.no_grad():
        for grade in range(5):
            z = torch.randn(8, LATENT_DIM, 1, 1, device=device)
            lbl = torch.full((8,), grade, dtype=torch.long, device=device)
            imgs = G(z, lbl).cpu()
            for i in range(8):
                ax = axes[grade][i]
                img = (imgs[i].permute(1,2,0).clamp(-1,1) + 1) / 2
                ax.imshow(img)
                ax.axis('off')
            axes[grade][0].set_ylabel(f'Grade {grade}\n{grade_names[grade]}', fontsize=9)

    fig.suptitle('Conditional GAN — 8 samples per DR grade', fontsize=13)
    plt.tight_layout()
    plt.savefig('outputs/cgan_per_grade.png', dpi=150)
    plt.show()
else:
    print('Train the cGAN first: python training/train_cgan.py')